In [ ]:
%run config_utility

In [ ]:
#parameter section
server="<<add your servername here>>.msit-database.fabric.microsoft.com"
database="configDatabase-xxxxx-xxxxx-xxxxx"

trackName = '<<add your track name here>>'
reportName = '<<add  your report name here>>'

StatementMeta(, 8a3b0802-03c9-4065-a781-95138753e02c, 33, Finished, Available, Finished)

Orchestrates the full DAX Performance & Query Analyzer workflow for a given Power BI report.
###### **Purpose:**  
Automates end-to-end execution of DAX queries for Import and Direct Lake modes, compares performance,
and logs results into SQL tables for later analysis and dashboarding.

###### **Workflow Steps:**
1. Fetch Source & Target workspace/dataset IDs from `metadata.vw_LoadTestDAX`.
2. Build Power BI and Fabric REST API URLs.
3. Retrieve test queries from metadata view.
4. Execute all DAX queries sequentially for Import (source) and Direct Lake (target).
5. Derive execution status columns.
6. Join source, target, and configuration results.
7. Persist the consolidated output to `logging.tbl_LoadTestDAX`.

In [ ]:
def main_workflow(server: str, database: str, trackName: str, reportName: str):
    config_jdbc_url = "jdbc:sqlserver://{};databaseName={};".format(server,database)

    config_df = fetch_metadata_config(trackName, reportName, config_jdbc_url)
    if config_df is None or config_df.rdd.isEmpty():
        print("[ERROR] Connection details for Source and Target environment not found; abort.")
        return

    try:
        sws, sds, tws, tds = extract_ids_from_config_df(config_df)
        source_url = f"https://api.powerbi.com/v1.0/myorg/groups/{sws}/datasets/{sds}/executeQueries"
        target_url = f"https://api.powerbi.com/v1.0/myorg/groups/{tws}/datasets/{tds}/executeQueries"
    except Exception as ex:
        print(f"[ERROR] Constructing URLs failed: {ex}")
        return

    test_df = fetch_test_queries(trackName, reportName, config_jdbc_url)
    if test_df is None or test_df.rdd.isEmpty():
        print("[ERROR] No queries definitions found; abort.")
        return

    src_res, tgt_res = run_queries_for_pages(test_df, source_url, target_url)
    if src_res is None or tgt_res is None:
        print("[ERROR] Query execution failed; abort.")
        return

    src_stat, tgt_stat = derive_status_columns(src_res, tgt_res)
    if src_stat is None or tgt_stat is None:
        print("[ERROR] Deriving status failed; abort.")
        return

    # Register temp views (explicit names)
    try:
        src_stat.createOrReplaceTempView("PowerBIModel_sequential_results")
        tgt_stat.createOrReplaceTempView("FabricModel_sequential_results")
        test_df.createOrReplaceTempView("config_df")
    except Exception as e:
        print(f"[WARN] Could not register views: {e}")

    runtime_id = str(uuid.uuid4())
    result_df = assemble_result_df(runtime_id)
    if result_df is None:
        print("[ERROR] Assembling result failed; abort.")
        return

    try:
        display(result_df)
    except Exception as e:
        print(f"[WARN] Cannot display result: {e}")

    log_and_persist_result(result_df, config_jdbc_url)

    print("[INFO] Workflow complete.")

In [ ]:
# To run:
main_workflow(server, database, trackName, reportName)